# Random Walk with Restart (RWR)

## Learning Objectives

1. **Define** RWR and its connection to personalized PageRank
2. **Implement** RWR via power iteration
3. **Explain** why RWR captures proximity better than shortest path
4. **Apply** RWR to node recommendation and community proximity

## What is RWR?

**Random Walk with Restart** (also called **Personalized PageRank**):

At each step, the walker either:
- With probability $1-\alpha$: follows a random out-edge
- With probability $\alpha$: **restarts** at the source node $s$

The steady-state probability vector $\mathbf{r}_s$ measures proximity to $s$:
- High $r_s(v)$: node $v$ is "close" to $s$ in the graph
- This captures **global structure**, not just direct neighbors

## Formal Definition

Steady state $\mathbf{r}_s$ satisfies:

$$\mathbf{r}_s = (1-\alpha) M \mathbf{r}_s + \alpha \mathbf{e}_s$$

where:
- $M$ = column-stochastic transition matrix
- $\mathbf{e}_s$ = unit vector at source node $s$ (the restart distribution)
- $\alpha$ = restart probability (typically 0.15)

**Solving:** $\mathbf{r}_s = \alpha (I - (1-\alpha)M)^{-1} \mathbf{e}_s$

**Computing:** power iteration — same as PageRank but with personalized restart.

In [1]:
from collections import defaultdict

def rwr(out_edges, nodes, source, alpha=0.15, n_iter=100):
    """
    Random Walk with Restart from source node.
    Returns proximity scores for all nodes.
    """
    n = len(nodes); idx = {v:i for i,v in enumerate(nodes)}
    out_deg = {u: len(out_edges.get(u,[])) for u in nodes}
    r = {v: (1.0 if v==source else 0.0) for v in nodes}

    for _ in range(n_iter):
        new_r = {v: 0.0 for v in nodes}
        for u in nodes:
            if out_deg.get(u,0) > 0:
                for v in out_edges.get(u,[]):
                    new_r[v] += (1-alpha) * r[u] / out_deg[u]
        # Restart
        new_r[source] += alpha
        diff = sum(abs(new_r[v]-r[v]) for v in nodes)
        r = new_r
        if diff < 1e-8: break
    return r

# Example graph
nodes = list(range(8))
out_e = {0:[1,2],1:[3],2:[3,4],3:[5],4:[5],5:[6,7],6:[7],7:[0]}
# Make undirected
for u in list(out_e.keys()):
    for v in out_e[u]:
        out_e.setdefault(v,[])
        if u not in out_e[v]: out_e[v].append(u)

prox = rwr(out_e, nodes, source=0, alpha=0.15)
print("RWR proximity scores from node 0:")
for v,score in sorted(prox.items(), key=lambda x:-x[1]):
    bar = '#'*int(score*80)
    print(f"  node {v}: {score:.4f}  {bar}")

RWR proximity scores from node 0:
  node 0: 0.2706  #####################
  node 2: 0.1350  ##########
  node 7: 0.1290  ##########
  node 5: 0.1214  #########
  node 3: 0.1099  ########
  node 1: 0.1078  ########
  node 4: 0.0640  #####
  node 6: 0.0623  ####


## RWR vs Shortest Path

**Shortest path:** uses only the single shortest route between nodes.
- Ignores redundant paths
- Sensitive to single edge removal
- No notion of "how many ways to get there"

**RWR:** accounts for **all paths** weighted by length and restart probability.
- Two short paths → higher score than one short path
- Robust to missing edges
- Naturally captures community structure (many internal paths → high scores within community)

In [2]:
from collections import deque

def bfs_shortest_path(adj, source):
    dist = {source: 0}; q = deque([source])
    while q:
        u = q.popleft()
        for v in adj.get(u,[]):
            if v not in dist:
                dist[v] = dist[u]+1; q.append(v)
    return dist

dist = bfs_shortest_path(out_e, 0)
prox = rwr(out_e, nodes, source=0, alpha=0.15)

print("Comparison: RWR proximity vs shortest path distance from node 0")
print(f"{'Node':>6} {'RWR':>8} {'ShortestPath':>14}")
for v in sorted(nodes):
    print(f"{v:>6} {prox[v]:>8.4f} {dist.get(v,'∞'):>14}")

Comparison: RWR proximity vs shortest path distance from node 0
  Node      RWR   ShortestPath
     0   0.2706              0
     1   0.1078              1
     2   0.1350              1
     3   0.1099              2
     4   0.0640              2
     5   0.1214              2
     6   0.0623              2
     7   0.1290              1


## Applications

**Recommendation:**
Given a user's history of liked items, run RWR from each liked item.
Aggregate proximity scores — recommend items with highest total proximity.

**Biology:** protein-protein interaction networks; run RWR from a known disease gene.
High-proximity proteins are candidate disease genes.

**Graph search:** query node $q$ in a knowledge graph; RWR from $q$ finds most related entities.

**Community detection:** run RWR from multiple seed nodes; nodes in the same community receive high scores from each other's seeds.

In [3]:
# Multi-source RWR for community detection
def multi_source_rwr(out_edges, nodes, sources, alpha=0.15):
    """Aggregate RWR from multiple sources."""
    agg = {v: 0.0 for v in nodes}
    for s in sources:
        r = rwr(out_edges, nodes, source=s, alpha=alpha)
        for v in nodes: agg[v] += r[v]
    return {v: agg[v]/len(sources) for v in nodes}

# Use nodes 0,1,2 as seeds → should rank nearby nodes highly
agg = multi_source_rwr(out_e, nodes, sources=[0,1,2])
print("Multi-source RWR (seeds: 0,1,2):")
for v,score in sorted(agg.items(), key=lambda x:-x[1]):
    bar = '#'*int(score*60)
    print(f"  node {v}: {score:.4f}  {bar}")

Multi-source RWR (seeds: 0,1,2):
  node 0: 0.1891  ###########
  node 2: 0.1743  ##########
  node 1: 0.1421  ########
  node 3: 0.1361  ########
  node 5: 0.1237  #######
  node 7: 0.1035  ######
  node 4: 0.0757  ####
  node 6: 0.0556  ###


## Summary

| Property | Shortest Path | RWR |
|----------|--------------|-----|
| Path count | Single | All paths weighted |
| Restart | N/A | Probability α per step |
| Output | Distance (integer) | Probability (float) |
| Robustness | Low | High |
| Community awareness | Low | **High** |

RWR = Personalized PageRank with a single source node.
The restart probability $\alpha$ controls the **locality** of the walk:
- High $\alpha$ → only immediate neighbors matter
- Low $\alpha$ → global structure captured